# Evaluation on Qwen/Qwen3-8B using BBEH benchmark for reasoning tasks

In [1]:
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 13.4 MB/s eta 0:00:00


In [2]:
import json
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics import classification_report

In [3]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
model_name = "Qwen/Qwen3-8B"

In [5]:
bitsandbytes_config = BitsAndBytesConfig(load_in_4bit=True,
                                         bnb_4bit_compute_dtype=torch.float16,
                                         bnb_4bit_quant_type='nf4',
                                         bnb_4bit_use_double_quant=True)

In [6]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model= AutoModelForCausalLM.from_pretrained(model_name,
                                             quantization_config=bitsandbytes_config,
                                             device_map="auto",
                                             offload_folder="offload")

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [7]:
def load_json(path):
  with open(path,"r") as f:
    data = json.load(f)

    df = pd.DataFrame(data['examples'])
    return df

In [8]:
def get_prompt(instruction,questions):
  prompts=[]
  for i in range(len(questions)):
    question = questions[i].split("Reply")[0]
    prompt = f"{instruction}\n\n{question}\n\nAnswer: "
    prompts.append(prompt)
  return prompts

In [9]:
def training(prompts,num_tokens=1):
  predictions=[]
  for i in range(len(prompts)):
    input_tokens = tokenizer(prompts[i],return_tensors="pt").to("cuda:0")
    output_tokens = model.generate(**input_tokens,max_new_tokens=num_tokens,
                                   do_sample=False,
                                   pad_token_id=tokenizer.eos_token_id,
                                   eos_token_id=tokenizer.eos_token_id)

    response = tokenizer.decode(output_tokens[0], skip_special_tokens=True).strip()
    pred_answer = response.split("Answer:")[1]
    print(f"Iteration {i}\n")
    print(f"Response: {pred_answer}")
    predictions.append(pred_answer)

  return predictions

## Task Casual Understanding

In [ ]:
dataset = load_json("/content/task_casual_understanding.json")

In [ ]:
dataset

,input,target
0,Question: Lauren and Jane work for the same co...,Yes
1,Question: Lauren and Jane work for the same co...,Yes
2,Question: The receptionist in the philosophy d...,Yes
3,Question: The receptionist in the philosophy d...,No
4,Question: Claire's parents bought her an old c...,Yes
...,...,...
195,Question: A test tube contains a mixture. Sam ...,Yes
196,Question: Reagent X and Y are both added to a ...,No
197,Question: There is a test tube on the left and...,Yes
198,Question: Sam is heating Reagent X in a test t...,Yes


In [ ]:
questions = dataset['input'].values.tolist()

In [ ]:
true_answers = dataset['target'].values.tolist()

In [ ]:
instruction = f"""Instruction: You are an AI assistant for casual understanding. Read the story and answer with exactly one word:
Yes (if you think majority of people will answer yes), No (if you think majority of people will answer no), or Ambiguous(If you think that majority of people do not know the answer) . Do not add explanations.
"""

In [ ]:
prompts = get_prompt(instruction,questions)

In [ ]:
prompts[0]

"Instruction: You are an AI assistant for casual understanding. Read the story and answer with exactly one word: \nYes (if you think majority of people will answer yes), No (if you think majority of people will answer no), or Ambiguous(If you think that majority of people do not know the answer) . Do not add explanations.\n\n\nQuestion: Lauren and Jane work for the same company. They each need to use a computer for work sometimes. Unfortunately, the computer isn't very powerful. If two people are logged on at the same time, it usually crashes. So the company decided to institute an official policy. It declared that Lauren would be the only one permitted to use the computer in the mornings and that Jane would be the only one permitted to use the computer in the afternoons. As expected, Lauren logged on the computer the next day at 9:00 am. But Jane decided to disobey the official policy. She also logged on at 9:00 am. The computer crashed immediately. Did Jane cause the computer to cras

In [ ]:
predictions = training(prompts,num_tokens=1)

Iteration 0

Response:   Yes
Iteration 1

Response:   Yes
Iteration 2

Response:   Yes
Iteration 3

Response:   Yes
Iteration 4

Response:   Yes
Iteration 5

Response:   No
Iteration 6

Response:   Yes
Iteration 7

Response:   Amb
Iteration 8

Response:   Alright
Iteration 9

Response:   Amb
Iteration 10

Response:   No
Iteration 11

Response:   No
Iteration 12

Response:   No
Iteration 13

Response:   Amb
Iteration 14

Response:   Yes
Iteration 15

Response:   Alright
Iteration 16

Response:   Let
Iteration 17

Response:   Amb
Iteration 18

Response:   Yes
Iteration 19

Response:   Yes
Iteration 20

Response:   Yes
Iteration 21

Response:   Alright
Iteration 22

Response:   Yes
Iteration 23

Response:   The
Iteration 24

Response:   Yes
Iteration 25

Response:   The
Iteration 26

Response:   Amb
Iteration 27

Response:   Amb
Iteration 28

Response:   Yes
Iteration 29

Response:   Yes
Iteration 30

Response:   Yes
Iteration 31

Response:   Yes
Iteration 32

Response:   No
Iteration 33


In [ ]:
predictions[:10]

['  Yes',
 '  Yes',
 '  Yes',
 '  Yes',
 '  Yes',
 '  No',
 '  Yes',
 '  Amb',
 '  Alright',
 '  Amb']

In [ ]:
predictions = [elem.strip() for elem in predictions]
predictions[:10]

['Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'No', 'Yes', 'Amb', 'Alright', 'Amb']

In [ ]:
predictions = [elem if elem=="Yes" or elem=="No" else "Ambiguous" for elem in predictions]
predictions[:10]

['Yes',
 'Yes',
 'Yes',
 'Yes',
 'Yes',
 'No',
 'Yes',
 'Ambiguous',
 'Ambiguous',
 'Ambiguous']

In [ ]:
print(classification_report(true_answers,predictions,zero_division=0))

              precision    recall  f1-score   support

   Ambiguous       0.20      0.38      0.26        45
          No       0.59      0.24      0.34        80
         Yes       0.42      0.47      0.44        75

    accuracy                           0.35       200
   macro avg       0.41      0.36      0.35       200
weighted avg       0.44      0.35      0.36       200



## Task Movie Recommendation

In [ ]:
dataset = load_json("/content/task_movie_recommendation.json")

In [ ]:
dataset

,input,target
0,Which option has more similar movies in terms ...,(E)
1,Which option has more similar movies in terms ...,(C)
2,Which option has more similar movies in terms ...,(C)
3,Which option has more similar movies in terms ...,(A)
4,Which option has more similar movies in terms ...,(I)
...,...,...
195,Which option has more similar movies in terms ...,(E)
196,Which option has more similar movies in terms ...,(E)
197,Which option has more similar movies in terms ...,(E)
198,Which option has more similar movies in terms ...,(A)


In [ ]:
dataset["target"].value_counts()

,count
target,
(A),26
(G),22
(H),21
(E),21
(J),20
(D),20
(C),18
(F),18
(B),17


In [ ]:
questions = dataset['input'].values.tolist()

In [ ]:
true_answers = dataset['target'].values.tolist()

In [ ]:
true_answers = [x.strip("()").upper() for x in true_answers]
true_answers[:10]

['E', 'C', 'C', 'A', 'I', 'D', 'B', 'F', 'D', 'H']

In [ ]:
instruction=f"""Instruction: You are an AI assistant that determines which group of movies is most similar.
Answer only with the one of the letters (**A**, **B**, **C**, **D**, **E**, **F**, **G**, **H**, **I**, or **J**) of the group that is most similar and then stop immediately. Do not write anything else.
Final output format: A,B,C,D,E,F,G,H,I or J
"""

In [ ]:
prompts = get_prompt(instruction,questions)

In [ ]:
prompts[1]

'Instruction: You are an AI assistant that determines which group of movies is most similar. \nAnswer only with the one of the letters (**A**, **B**, **C**, **D**, **E**, **F**, **G**, **H**, **I**, or **J**) of the group that is most similar and then stop immediately. Do not write anything else.\nFinal output format: A,B,C,D,E,F,G,H,I or J\n\n\nWhich option has more similar movies in terms of whether a group of people will like them or not?\nOptions:\n(A) Street Fighter II The Animated Movie, The Sixth Sense, The Matrix, The Shawshank Redemption, Terminator 2 Judgment Day\n(B) The Boy Who Could Fly, The Sheltering Sky, Forrest Gump, Terminator 2 Judgment Day, The Matrix\n(C) Forrest Gump, The Shawshank Redemption, Terminator 2 Judgment Day, The Matrix, The Sixth Sense\n(D) The Boy Who Could Fly, The Sheltering Sky, Street Fighter II The Animated Movie, The Sixth Sense, Terminator 2 Judgment Day\n(E) The Matrix, The Boy Who Could Fly, Forrest Gump, The Shawshank Redemption, The Sixth S

In [ ]:
predictions = training(prompts,num_tokens=1)

Iteration 0

Response:   J
Iteration 1

Response:   C
Iteration 2

Response:   To
Iteration 3

Response:   To
Iteration 4

Response:   I
Iteration 5

Response:   D
Iteration 6

Response:   G
Iteration 7

Response:   F
Iteration 8

Response:   D
Iteration 9

Response:   G
Iteration 10

Response:   I
Iteration 11

Response:   G
Iteration 12

Response:   I
Iteration 13

Response:   I
Iteration 14

Response:   G
Iteration 15

Response:   The
Iteration 16

Response:   D
Iteration 17

Response:   E
Iteration 18

Response:   G
Iteration 19

Response:   I
Iteration 20

Response:   To
Iteration 21

Response:   G
Iteration 22

Response:   H
Iteration 23

Response:   H
Iteration 24

Response:   D
Iteration 25

Response:   I
Iteration 26

Response:   D
Iteration 27

Response:   H
Iteration 28

Response:   J
Iteration 29

Response:   J
Iteration 30

Response:   I
Iteration 31

Response:   G
Iteration 32

Response:   D
Iteration 33

Response:   F
Iteration 34

Response:   I
Iteration 35

Response:  

In [ ]:
predictions[:10]

['  J', '  C', '  To', '  To', '  I', '  D', '  G', '  F', '  D', '  G']

In [ ]:
predictions = [elem.strip() for elem in predictions]
predictions[:10]

['J', 'C', 'To', 'To', 'I', 'D', 'G', 'F', 'D', 'G']

In [ ]:
predictions = [elem if elem  in list("ABCDEFGHIJ") else "Unknown" for elem in predictions]
predictions[:10]

['J', 'C', 'Unknown', 'Unknown', 'I', 'D', 'G', 'F', 'D', 'G']

In [ ]:
print(classification_report(true_answers,predictions,zero_division=0))

              precision    recall  f1-score   support

           A       0.00      0.00      0.00        26
           B       0.00      0.00      0.00        17
           C       1.00      0.06      0.11        18
           D       0.47      0.40      0.43        20
           E       1.00      0.05      0.09        21
           F       0.38      0.44      0.41        18
           G       0.39      0.64      0.48        22
           H       0.40      0.38      0.39        21
           I       0.25      0.88      0.38        17
           J       0.39      0.45      0.42        20
     Unknown       0.00      0.00      0.00         0

    accuracy                           0.32       200
   macro avg       0.39      0.30      0.25       200
weighted avg       0.42      0.32      0.27       200



## Task Disambiguation QA

In [10]:
dataset = load_json("/content/task_disambiguationQA.json")

In [11]:
dataset['target'].value_counts()

,count
target,
(B),25
(D),24
(C),23
(A),22
(E),17
(F),7
(G),1
(H),1


In [12]:
questions = dataset['input'].values.tolist()

In [13]:
true_answers = dataset['target'].values.tolist()

In [14]:
true_answers = [x.strip("()").upper() for x in true_answers]
true_answers[:10]

['A', 'D', 'E', 'B', 'A', 'B', 'B', 'A', 'C', 'D']

In [15]:
instruction = f"""You are an AI assistant solving a pronoun disambiguation task.

For each sentence:
- Identify what the pronoun(s) most likely refer to based on common sense.
- If more than one interpretation is equally plausible, select “Ambiguous”.

Rules:
- Do not assume information that is not stated.
- Do not infer intent beyond what is commonly assumed.
- Choose exactly one answer from (A)–(H) based on available letters.

Final answer format:
(A), (B), (C), (D), (E), (H), (G), or (F)
"""

In [16]:
prompts = get_prompt(instruction,questions)

In [17]:
prompts[1]

'You are an AI assistant solving a pronoun disambiguation task.\n\nFor each sentence:\n- Identify what the pronoun(s) most likely refer to based on common sense.\n- If more than one interpretation is equally plausible, select “Ambiguous”.\n\nRules:\n- Do not assume information that is not stated.\n- Do not infer intent beyond what is commonly assumed.\n- Choose exactly one answer from (A)–(H) based on available letters.\n\nFinal answer format:\n(A), (B), (C), (D), (E), (H), (G), or (F)\n\n\nHere is a sentence with pronoun(s) whose antecedent(s) can either be derived from the context or is ambiguous.\nSentence: The designer called the janitor and asked him to clean the room as soon as possible, but he went to fetch a cup of coffee first and then returned to his desk.\nWhich of the following options correctly explains the antecedent(s) of the pronoun(s)?\n(A) Designer is cleaning the room as soon as possible and got coffee.\n(B) Janitor is cleaning the room as soon as possible and got co

In [27]:
predictions = training(prompts,num_tokens=3)

Iteration 0

Response:   (E)
Iteration 1

Response:   (E)
Iteration 2

Response:   (E)
Iteration 3

Response:   (E)
Iteration 4

Response:   (D)
Iteration 5

Response:   (E)
Iteration 6

Response:   (C)
Iteration 7

Response:   (D)
Iteration 8

Response:   (B)
Iteration 9

Response:   (D)
Iteration 10

Response:   (C)
Iteration 11

Response:   (E)
Iteration 12

Response:   (C)
Iteration 13

Response:   (B)
Iteration 14

Response:   (C)
Iteration 15

Response:   (E)
Iteration 16

Response:   (D)
Iteration 17

Response:   (B)
Iteration 18

Response:   (E)
Iteration 19

Response:   (C)
Iteration 20

Response:   (B)
Iteration 21

Response:   (D)
Iteration 22

Response:   (D)
Iteration 23

Response:   (C)
Iteration 24

Response:   (C)
Iteration 25

Response:   (D)
Iteration 26

Response:   (E)
Iteration 27

Response:   Let's analyze
Iteration 28

Response:   Let's analyze
Iteration 29

Response:   Let's analyze
Iteration 30

Response:   (A),
Iteration 31

Response:   (C)
Iteration 32

Respo

In [28]:
predictions[:10]

['  (E)',
 '  (E)',
 '  (E)',
 '  (E)',
 '  (D)',
 '  (E)',
 '  (C)',
 '  (D)',
 '  (B)',
 '  (D)']

In [29]:
predictions = [x.strip() for x in predictions]
predictions[:10]

['(E)', '(E)', '(E)', '(E)', '(D)', '(E)', '(C)', '(D)', '(B)', '(D)']

In [30]:
predictions = [x.strip("()").upper() for x in predictions]
predictions

['E',
 'E',
 'E',
 'E',
 'D',
 'E',
 'C',
 'D',
 'B',
 'D',
 'C',
 'E',
 'C',
 'B',
 'C',
 'E',
 'D',
 'B',
 'E',
 'C',
 'B',
 'D',
 'D',
 'C',
 'C',
 'D',
 'E',
 "LET'S ANALYZE",
 "LET'S ANALYZE",
 "LET'S ANALYZE",
 'A),',
 'C',
 "LET'S ANALYZE",
 'E',
 'E',
 "LET'S ANALYZE",
 'E',
 "LET'S ANALYZE",
 'F',
 'E',
 "LET'S ANALYZE",
 "LET'S ANALYZE",
 'B',
 'E',
 'C',
 "LET'S ANALYZE",
 "LET'S ANALYZE",
 "LET'S ANALYZE",
 "LET'S ANALYZE",
 'B',
 'F',
 'E',
 'F',
 'E',
 'D',
 'D',
 'C',
 "LET'S ANALYZE",
 'E',
 'D',
 "LET'S ANALYZE",
 'E',
 'E',
 'B',
 'B',
 "LET'S ANALYZE",
 'D',
 'C',
 'E',
 'D',
 'D',
 'A),',
 'E',
 'C',
 'B',
 'C',
 'D',
 'B',
 'E',
 'B',
 'C',
 'E',
 'B',
 'A),',
 'D',
 'D',
 'C',
 'C',
 'E',
 'B',
 'F',
 "LET'S ANALYZE",
 'E',
 'E',
 'E',
 'E',
 'C',
 'C',
 'C',
 'B',
 'D',
 'C',
 'E',
 'E',
 'E',
 'E',
 'D',
 'E',
 'E',
 "LET'S ANALYZE",
 'F',
 'F',
 'C',
 "LET'S ANALYZE",
 'F',
 'D',
 'E',
 'B',
 'E',
 'E']

In [31]:
predictions_new = [x.strip("),") for x in predictions]
predictions_new

['E',
 'E',
 'E',
 'E',
 'D',
 'E',
 'C',
 'D',
 'B',
 'D',
 'C',
 'E',
 'C',
 'B',
 'C',
 'E',
 'D',
 'B',
 'E',
 'C',
 'B',
 'D',
 'D',
 'C',
 'C',
 'D',
 'E',
 "LET'S ANALYZE",
 "LET'S ANALYZE",
 "LET'S ANALYZE",
 'A',
 'C',
 "LET'S ANALYZE",
 'E',
 'E',
 "LET'S ANALYZE",
 'E',
 "LET'S ANALYZE",
 'F',
 'E',
 "LET'S ANALYZE",
 "LET'S ANALYZE",
 'B',
 'E',
 'C',
 "LET'S ANALYZE",
 "LET'S ANALYZE",
 "LET'S ANALYZE",
 "LET'S ANALYZE",
 'B',
 'F',
 'E',
 'F',
 'E',
 'D',
 'D',
 'C',
 "LET'S ANALYZE",
 'E',
 'D',
 "LET'S ANALYZE",
 'E',
 'E',
 'B',
 'B',
 "LET'S ANALYZE",
 'D',
 'C',
 'E',
 'D',
 'D',
 'A',
 'E',
 'C',
 'B',
 'C',
 'D',
 'B',
 'E',
 'B',
 'C',
 'E',
 'B',
 'A',
 'D',
 'D',
 'C',
 'C',
 'E',
 'B',
 'F',
 "LET'S ANALYZE",
 'E',
 'E',
 'E',
 'E',
 'C',
 'C',
 'C',
 'B',
 'D',
 'C',
 'E',
 'E',
 'E',
 'E',
 'D',
 'E',
 'E',
 "LET'S ANALYZE",
 'F',
 'F',
 'C',
 "LET'S ANALYZE",
 'F',
 'D',
 'E',
 'B',
 'E',
 'E']

In [32]:
predictions = [elem if elem  in list("ABCDEFGH") else "Unknown" for elem in predictions_new]
predictions

['E',
 'E',
 'E',
 'E',
 'D',
 'E',
 'C',
 'D',
 'B',
 'D',
 'C',
 'E',
 'C',
 'B',
 'C',
 'E',
 'D',
 'B',
 'E',
 'C',
 'B',
 'D',
 'D',
 'C',
 'C',
 'D',
 'E',
 'Unknown',
 'Unknown',
 'Unknown',
 'A',
 'C',
 'Unknown',
 'E',
 'E',
 'Unknown',
 'E',
 'Unknown',
 'F',
 'E',
 'Unknown',
 'Unknown',
 'B',
 'E',
 'C',
 'Unknown',
 'Unknown',
 'Unknown',
 'Unknown',
 'B',
 'F',
 'E',
 'F',
 'E',
 'D',
 'D',
 'C',
 'Unknown',
 'E',
 'D',
 'Unknown',
 'E',
 'E',
 'B',
 'B',
 'Unknown',
 'D',
 'C',
 'E',
 'D',
 'D',
 'A',
 'E',
 'C',
 'B',
 'C',
 'D',
 'B',
 'E',
 'B',
 'C',
 'E',
 'B',
 'A',
 'D',
 'D',
 'C',
 'C',
 'E',
 'B',
 'F',
 'Unknown',
 'E',
 'E',
 'E',
 'E',
 'C',
 'C',
 'C',
 'B',
 'D',
 'C',
 'E',
 'E',
 'E',
 'E',
 'D',
 'E',
 'E',
 'Unknown',
 'F',
 'F',
 'C',
 'Unknown',
 'F',
 'D',
 'E',
 'B',
 'E',
 'E']

In [33]:
print(classification_report(true_answers,predictions,zero_division=0))

              precision    recall  f1-score   support

           A       0.33      0.05      0.08        22
           B       0.40      0.24      0.30        25
           C       0.52      0.48      0.50        23
           D       0.53      0.42      0.47        24
           E       0.22      0.47      0.30        17
           F       0.43      0.43      0.43         7
           G       0.00      0.00      0.00         1
           H       0.00      0.00      0.00         1
     Unknown       0.00      0.00      0.00         0

    accuracy                           0.33       120
   macro avg       0.27      0.23      0.23       120
weighted avg       0.41      0.33      0.33       120



## Task: Boolean Expressions

In [34]:
dataset = load_json("/content/task_boolean_expressions.json")

In [35]:
dataset

,input,target
0,"From the following five expressions, only one ...",(E)
1,"From the following five expressions, only one ...",(C)
2,"From the following five expressions, only one ...",(E)
3,"From the following five expressions, only one ...",(D)
4,"From the following five expressions, only one ...",(C)
...,...,...
195,"From the following five expressions, only one ...",(A)
196,"From the following five expressions, only one ...",(B)
197,"From the following five expressions, only one ...",(B)
198,"From the following five expressions, only one ...",(C)


In [36]:
dataset["target"].value_counts()

,count
target,
(E),43
(B),41
(C),40
(A),39
(D),37


In [37]:
questions = dataset["input"].values.tolist()

In [38]:
true_answers = dataset["target"].values.tolist()

In [39]:
true_answers = [x.strip("()").upper() for x in true_answers]
true_answers[:10]

['E', 'C', 'E', 'D', 'C', 'D', 'C', 'D', 'A', 'A']

In [40]:
instruction=f"""You are evaluating boolean expressions.

Exactly one option is True.

Output format rules:
- Output ONLY one of the following exact strings:
(A)
(B)
(C)
(D)
(E)

Do NOT use LaTeX.
Do NOT use special symbols.
Do NOT add explanations.
Do NOT add whitespace or extra characters.
"""

In [41]:
prompts = get_prompt(instruction,questions)

In [42]:
prompts[0]

'You are evaluating boolean expressions.\n\nExactly one option is True.\n\nOutput format rules:\n- Output ONLY one of the following exact strings:\n(A)\n(B)\n(C)\n(D)\n(E)\n\nDo NOT use LaTeX.\nDo NOT use special symbols.\nDo NOT add explanations.\nDo NOT add whitespace or extra characters.\n\n\nFrom the following five expressions, only one evaluates to true. Which one is it?\n\n(A) not (not (not (not (False) or (-5 - (1 / -5) is greater than -7)) or (not (False) or (-5 * -10 + -5 * -6 is greater than -4 * 6))) or (not (not (False) or (False)) or (not not not (-3 - (-2 / 4) is greater than -6) and not not (The capital of India is Mumbai.)))) and (not (not (not (The capital of the Nigeria is Abuja.) or (-7 - (-4 / 7) > 4)) or (not (-6 - (-7 / -5) > -6) and (True))) and (not not ((-1 * 9 is less than or equal to -6) or (-4 * 10 + -1 * -7 <= 9 * 10)) or not not not (not not (6 * -2 is less than or equal to 4) or not not not (-5 * 9 + 6 * -3 <= -9 * 6))))\n(B) ((not (not not not (not (not 

In [43]:
predictions = training(prompts,num_tokens=3)

Iteration 0

Response:   (D)
Iteration 1

Response:   (E)
Iteration 2

Response:   (E)
Iteration 3

Response:   (E)
Iteration 4

Response:   Let's break
Iteration 5

Response:   (D)
Iteration 6

Response:   (D)
Iteration 7

Response:   (D)
Iteration 8

Response:   Let's break
Iteration 9

Response:   (D)
Iteration 10

Response:   Let's break
Iteration 11

Response:   (D)
Iteration 12

Response:   (E)
Iteration 13

Response:   (D)
Iteration 14

Response:   (D)
Iteration 15

Response:   (E)
Iteration 16

Response:   (E)
Iteration 17

Response:   (D)
Iteration 18

Response:   (D)
Iteration 19

Response:   (E)
Iteration 20

Response:   (E)
Iteration 21

Response:   (E)
Iteration 22

Response:   (E)
Iteration 23

Response:   (D)
Iteration 24

Response:   (D)
Iteration 25

Response:   Let's break
Iteration 26

Response:   (D)
Iteration 27

Response:   (D)
Iteration 28

Response:   (D)
Iteration 29

Response:   (D)
Iteration 30

Response:   (D)
Iteration 31

Response:   (E)
Iteration 32

Resp

In [44]:
predictions[:10]

['  (D)',
 '  (E)',
 '  (E)',
 '  (E)',
 "  Let's break",
 '  (D)',
 '  (D)',
 '  (D)',
 "  Let's break",
 '  (D)']

In [45]:
predictions = [x.strip() for x in predictions]
predictions[:10]

['(D)',
 '(E)',
 '(E)',
 '(E)',
 "Let's break",
 '(D)',
 '(D)',
 '(D)',
 "Let's break",
 '(D)']

In [46]:
predictions = [x.strip("()").upper() for x in predictions]
predictions

['D',
 'E',
 'E',
 'E',
 "LET'S BREAK",
 'D',
 'D',
 'D',
 "LET'S BREAK",
 'D',
 "LET'S BREAK",
 'D',
 'E',
 'D',
 'D',
 'E',
 'E',
 'D',
 'D',
 'E',
 'E',
 'E',
 'E',
 'D',
 'D',
 "LET'S BREAK",
 'D',
 'D',
 'D',
 'D',
 'D',
 'E',
 'D',
 'E',
 'D',
 'D',
 'E',
 'D',
 'D',
 'D',
 'D',
 'D',
 'D',
 'D',
 'D',
 'D',
 'D',
 'E',
 'D',
 "LET'S BREAK",
 'E',
 'D',
 'D',
 'D',
 'D',
 'D',
 'D',
 'A',
 'E',
 'D',
 'D',
 'D',
 'E',
 'D',
 'D',
 'D',
 'D',
 'D',
 'D',
 'D',
 'D',
 'A',
 'E',
 'D',
 'D',
 'E',
 'D',
 'D',
 'D',
 'D',
 'D',
 'E',
 'E',
 'E',
 'E',
 'D',
 'D',
 'E',
 'D',
 'D',
 'D',
 'D',
 'D',
 'D',
 'D',
 'D',
 'E',
 'E',
 'D',
 'A',
 'D',
 'D',
 'D',
 'D',
 "LET'S BREAK",
 'E',
 'D',
 'D',
 'D',
 'E',
 'D',
 'D',
 'D',
 'E',
 'D',
 'D',
 'A',
 'D',
 'E',
 'D',
 'D',
 'E',
 'D',
 'E',
 'E',
 'D',
 'D',
 'D',
 'E',
 'D',
 'E',
 'E',
 'D',
 'D',
 'D',
 'D',
 'D',
 'D',
 'D',
 'D',
 'E',
 'D',
 'D',
 'E',
 'D',
 'D',
 'D',
 'D',
 'D',
 'D',
 'D',
 'D',
 'E',
 'E',
 'D',
 'D',
 'D'

In [47]:
predictions = [elem if elem  in list("ABCDE") else "Unknown" for elem in predictions]
predictions

['D',
 'E',
 'E',
 'E',
 'Unknown',
 'D',
 'D',
 'D',
 'Unknown',
 'D',
 'Unknown',
 'D',
 'E',
 'D',
 'D',
 'E',
 'E',
 'D',
 'D',
 'E',
 'E',
 'E',
 'E',
 'D',
 'D',
 'Unknown',
 'D',
 'D',
 'D',
 'D',
 'D',
 'E',
 'D',
 'E',
 'D',
 'D',
 'E',
 'D',
 'D',
 'D',
 'D',
 'D',
 'D',
 'D',
 'D',
 'D',
 'D',
 'E',
 'D',
 'Unknown',
 'E',
 'D',
 'D',
 'D',
 'D',
 'D',
 'D',
 'A',
 'E',
 'D',
 'D',
 'D',
 'E',
 'D',
 'D',
 'D',
 'D',
 'D',
 'D',
 'D',
 'D',
 'A',
 'E',
 'D',
 'D',
 'E',
 'D',
 'D',
 'D',
 'D',
 'D',
 'E',
 'E',
 'E',
 'E',
 'D',
 'D',
 'E',
 'D',
 'D',
 'D',
 'D',
 'D',
 'D',
 'D',
 'D',
 'E',
 'E',
 'D',
 'A',
 'D',
 'D',
 'D',
 'D',
 'Unknown',
 'E',
 'D',
 'D',
 'D',
 'E',
 'D',
 'D',
 'D',
 'E',
 'D',
 'D',
 'A',
 'D',
 'E',
 'D',
 'D',
 'E',
 'D',
 'E',
 'E',
 'D',
 'D',
 'D',
 'E',
 'D',
 'E',
 'E',
 'D',
 'D',
 'D',
 'D',
 'D',
 'D',
 'D',
 'D',
 'E',
 'D',
 'D',
 'E',
 'D',
 'D',
 'D',
 'D',
 'D',
 'D',
 'D',
 'D',
 'E',
 'E',
 'D',
 'D',
 'D',
 'Unknown',
 'D',
 'D'

In [48]:
print(classification_report(true_answers,predictions,zero_division=0))

              precision    recall  f1-score   support

           A       0.00      0.00      0.00        39
           B       0.00      0.00      0.00        41
           C       0.00      0.00      0.00        40
           D       0.20      0.70      0.31        37
           E       0.20      0.26      0.22        43
     Unknown       0.00      0.00      0.00         0

    accuracy                           0.18       200
   macro avg       0.07      0.16      0.09       200
weighted avg       0.08      0.18      0.11       200

